# Notebook 1.6: The Mandel Q-Parameter and Non-Classicality Criteria
### Module 1: Single-Mode Quantum States of Light
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_1_states_of_light/08_mandel_q_parameter.ipynb)

---

### Prerequisites
- Notebook 1.1: Fock States, Ladder Operators, and Truncation Artifacts
- Notebook 1.2: Coherent States, Displacement, and Poissonian Statistics
- Notebook 1.3: Squeezed Vacuum and Quadrature Noise Reduction
- Notebook 1.4: Displaced Squeezed States and Phase Space Interference
- Notebook 1.5: Thermal States and Mixed Radiation Fields

### Pedagogical Learning Objectives
1. **The Mandel $Q$-Parameter & Fano Factor:** Formulate the definitions of $Q$ and $F = \text{Var}(n)/\langle n \rangle$, understanding their mathematical limits ($Q \ge -1$, $F \ge 0$).
2. **The Classical Statistical Boundary:** Prove why $Q \ge 0$ for all classical mixtures of electromagnetic waves using the Glauber-Sudarshan $P$-representation, demonstrating that $Q < 0$ is an unequivocal criterion for quantum non-classical light.
3. **Connection to Second-Order Coherence:** Derive the exact fundamental identity $Q = \langle \hat{n} \rangle [g^{(2)}(0) - 1]$, linking photon counting variance to photon anti-bunching.
4. **Sufficient vs. Necessary Non-Classicality:** Understand why $Q < 0$ is a sufficient condition for non-classicality, but not necessary (e.g. squeezed vacuum has $Q > 0$ despite possessing strong quadrature non-classicality).
5. **Cross-State Benchmarking:** Systematically compute and plot $Q$ across Fock states, coherent states, thermal states, squeezed vacuum, and amplitude-squeezed light as a function of mean photon number.
6. **Numerical Implementation & QuTiP Verification:** Construct modular moment-calculating tools for state vectors and density matrices, benchmarking with QuTiP.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider, Dropdown
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. The Mandel $Q$-Parameter: Definition and Classical Boundary

In 1979, **Leonard Mandel** introduced a dimensionless normalized measure to characterize deviations of photon counting statistics from the standard Poisson distribution:

$$
Q \equiv \frac{\langle (\Delta \hat{n})^2 \rangle - \langle \hat{n} \rangle}{\langle \hat{n} \rangle} = \frac{\text{Var}(n)}{\langle n \rangle} - 1 = F - 1
$$

where:
- $\langle \hat{n} \rangle = \langle \hat{a}^\dagger \hat{a} \rangle$ is the mean photon number.
- $\text{Var}(n) = \langle \hat{n}^2 \rangle - \langle \hat{n} \rangle^2$ is the photon number variance.
- $F = \frac{\text{Var}(n)}{\langle n \rangle}$ is the **Fano factor**.

---

### Classification of Light Regimes

Depending on the sign of $Q$ (or whether $F \gtrless 1$):

1. **Sub-Poissonian Light ($-1 \le Q < 0$, $0 \le F < 1$):**
   The photon number variance is smaller than the mean: $\text{Var}(n) < \langle n \rangle$. Photons arrive more regularly spaced than a random Poisson process. **Strictly non-classical quantum light!**
2. **Poissonian Light ($Q = 0$, $F = 1$):**
   The variance equals the mean: $\text{Var}(n) = \langle n \rangle$. Characteristic of an ideal classical laser operating well above threshold (coherent state $|\alpha\rangle$).
3. **Super-Poissonian Light ($Q > 0$, $F > 1$):**
   The variance exceeds the mean: $\text{Var}(n) > \langle n \rangle$. Characteristic of thermal blackbody radiation, chaotic discharge lamps, or phase-fluctuating fields.

---

### Proof: Why $Q < 0$ Forbids a Classical Explanation

Why is $Q < 0$ impossible for classical light?

In classical statistical optics, an optical field is described by a fluctuating complex amplitude $\alpha(t) = I(t)^{1/2} e^{i\phi(t)}$. According to the **Glauber-Sudarshan $P$-representation**, any quantum density operator can be expressed as:

$$
\hat{\rho} = \int P(\alpha) |\alpha\rangle\langle \alpha| \, d^2\alpha
$$

Using normal ordering and the coherent state property $\hat{a}|\alpha\rangle = \alpha |\alpha\rangle$:
$$
\langle \hat{n} \rangle = \mathrm{Tr}(\hat{\rho} \hat{a}^\dagger \hat{a}) = \int P(\alpha) |\alpha|^2 \, d^2\alpha = \overline{I}
$$
$$
\langle :(\Delta \hat{n})^2: \rangle = \langle (\hat{a}^\dagger)^2 \hat{a}^2 \rangle - \langle \hat{a}^\dagger \hat{a} \rangle^2 = \int P(\alpha) |\alpha|^4 \, d^2\alpha - \overline{I}^2 = \overline{I^2} - \overline{I}^2 = \overline{(I - \overline{I})^2}
$$

Using the commutator $[\hat{a}, \hat{a}^\dagger] = 1$, the quantum variance expands to:
$$
\langle (\Delta \hat{n})^2 \rangle = \langle \hat{n} \rangle + \langle :(\Delta \hat{n})^2: \rangle = \langle \hat{n} \rangle + \int P(\alpha) (|\alpha|^2 - \overline{I})^2 \, d^2\alpha
$$

Therefore, the Mandel $Q$-parameter is:
$$
Q = \frac{\int P(\alpha) (|\alpha|^2 - \overline{I})^2 \, d^2\alpha}{\int P(\alpha) |\alpha|^2 \, d^2\alpha}
$$

If the field is classical, $P(\alpha)$ is a bona fide probability density function satisfying $P(\alpha) \ge 0$. The integrand $(|\alpha|^2 - \overline{I})^2 \ge 0$ is non-negative everywhere, which **strictly requires**:

$$
Q_{\text{classical}} \ge 0
$$

**Conclusion:** Observing $Q < 0$ proves that $P(\alpha)$ takes negative values or is more singular than a Dirac delta function. Sub-Poissonian light has **no classical electromagnetic wave description**!


In [ ]:
# Comprehensive Statistical Suite: Calculating Moments, Fano Factor, and Mandel Q

def quantum_moments(state, N):
    """Computes mean photon number, variance, Fano factor, and Mandel Q from state vector or density matrix."""
    n_diag = np.arange(N, dtype=np.float64)
    n_op = np.diag(n_diag)
    n2_op = np.diag(n_diag**2)
    
    if state.ndim == 1:
        # State vector |psi>
        norm = np.linalg.norm(state)
        psi = state / norm
        mean_n = np.real(np.vdot(psi, n_op @ psi))
        mean_n2 = np.real(np.vdot(psi, n2_op @ psi))
    else:
        # Density matrix rho
        tr = np.real(np.trace(state))
        rho = state / tr
        mean_n = np.real(np.trace(rho @ n_op))
        mean_n2 = np.real(np.trace(rho @ n2_op))
        
    var_n = mean_n2 - mean_n**2
    
    if mean_n < 1e-12:
        fano = 0.0
        mandel_q = -1.0 if var_n < 1e-12 else 0.0
    else:
        fano = var_n / mean_n
        mandel_q = fano - 1.0
        
    return {
        "mean_n": mean_n,
        "var_n": var_n,
        "std_n": np.sqrt(max(0.0, var_n)),
        "fano_factor": fano,
        "mandel_q": mandel_q
    }

# Analytical functions for comparison
def mandel_q_fock(n):
    return -1.0 if n > 0 else 0.0

def mandel_q_coherent(alpha):
    return 0.0

def mandel_q_thermal(n_bar):
    return float(n_bar)

def mandel_q_squeezed_vac(r):
    # Squeezed vacuum: <n> = sinh^2(r), Var(n) = 2 sinh^2(r) cosh^2(r)
    # Q = [2 sinh^2(r) cosh^2(r) - sinh^2(r)] / sinh^2(r) = 2 cosh^2(r) - 1 = cosh(2r) = 1 + 2*n_bar
    return float(np.cosh(2 * r))

# Numerical benchmark test
N_dim = 60
n_fock = 5
fock_state = np.zeros(N_dim, dtype=np.complex128)
fock_state[n_fock] = 1.0

res_fock = quantum_moments(fock_state, N_dim)
print("--- Fock State |5> Benchmark ---")
print(f"Mean n:   {res_fock['mean_n']:.4f} (Expected: 5.0000)")
print(f"Var(n):   {res_fock['var_n']:.4e} (Expected: 0.0000)")
print(f"Fano F:   {res_fock['fano_factor']:.4e} (Expected: 0.0000)")
print(f"Mandel Q: {res_fock['mandel_q']:.4f} (Expected: -1.0000, Absolute Non-classical Limit)")
assert abs(res_fock['mandel_q'] - (-1.0)) < 1e-8, "Fock state Mandel Q failed!"


## 2. Sufficient vs. Necessary Non-Classicality Criteria

A central nuance in quantum optics is understanding the hierarchy of non-classicality witnesses:

$$
\text{Sub-Poissonian Statistics } (Q < 0) \quad \subset \quad \text{Non-Classical Quantum Light}
$$

### Why $Q < 0$ is Sufficient, but NOT Necessary:
- **Sufficient:** If $Q < 0$, the state is guaranteed to be non-classical beyond all doubt.
- **Not Necessary:** A quantum state can possess profound non-classical features (quadrature squeezing, quantum entanglement, negative Wigner functions) while having $Q \ge 0$!

---

### The Squeezed Vacuum Paradox: Highly Non-Classical, yet Super-Poissonian!
Consider a squeezed vacuum state $|0, r\rangle$:
1. **Quadrature Noise:** $\langle (\Delta \hat{X}_1)^2 \rangle = \frac{1}{4}e^{-2r} < \frac{1}{4}$ (Sub-shot-noise, purely quantum).
2. **Wigner Negativity / Phase Squeezing:** Unmistakably quantum mechanical.
3. **Mandel $Q$-Parameter:**
   $$
   \langle \hat{n} \rangle = \sinh^2 r = \bar{n}
   $$
   $$
   \text{Var}(n) = 2\sinh^2 r \cosh^2 r = 2\bar{n}(\bar{n} + 1)
   $$
   $$
   Q_{\text{sq\_vac}} = \frac{2\bar{n}(\bar{n}+1) - \bar{n}}{\bar{n}} = 2\bar{n} + 1 = \cosh 2r > 0
   $$

The squeezed vacuum is **super-Poissonian** ($Q > 0$) for any finite squeezing ($r > 0$)! Why? Because parametric down-conversion generates photons strictly in correlated pairs ($|0\rangle, |2\rangle, |4\rangle \dots$). The absence of odd photon states creates huge variance in photon counting, resulting in super-Poissonian statistics despite its non-classical nature.

---

### Amplitude Squeezing: Achieving $Q < 0$ with Continuous Fields
To obtain sub-Poissonian light ($Q < 0$) from a squeezed state, one must **displace the squeezed state** along its squeezed axis:
$$
|\alpha, r\rangle = \hat{D}(\alpha) \hat{S}(r) |0\rangle \quad (\alpha \in \mathbb{R}, r > 0)
$$
Here:
$$
\langle (\Delta \hat{n})^2 \rangle = \alpha^2 e^{-2r} + 2\sinh^2 r \cosh^2 r
$$
$$
Q = \frac{\alpha^2 (e^{-2r} - 1) + \sinh^2 r \cosh 2r}{\alpha^2 + \sinh^2 r}
$$
When the coherent amplitude is large ($\alpha^2 \gg \sinh^2 r$), $Q \approx e^{-2r} - 1 < 0$!


In [ ]:
# Comprehensive Multi-State Mandel Q Comparison as a Function of Mean Photons <n>

n_sweep = np.linspace(0.1, 15.0, 150)

# 1. Fock State |n>: Q = -1 (constant)
q_fock = -np.ones_like(n_sweep)

# 2. Coherent State |alpha>: Q = 0 (constant)
q_coherent = np.zeros_like(n_sweep)

# 3. Thermal State: Q = n_bar
q_thermal = n_sweep.copy()

# 4. Squeezed Vacuum: Q = 1 + 2*n_bar
q_sq_vac = 1.0 + 2.0 * n_sweep

# 5. Amplitude Squeezed State: fixed squeezing r = 0.6 (~5.2 dB), sweeping displacement alpha
# n_bar = alpha^2 + sinh^2(r)  ==>  alpha^2 = n_bar - sinh^2(r)
r_fixed = 0.6
n_sq_vac = np.sinh(r_fixed)**2
valid_mask = n_sweep > n_sq_vac

q_amp_squeezed = np.zeros_like(n_sweep)
for i, n_val in enumerate(n_sweep):
    if n_val > n_sq_vac:
        alpha_sq = n_val - n_sq_vac
        var_n = alpha_sq * np.exp(-2 * r_fixed) + 2 * np.sinh(r_fixed)**2 * np.cosh(r_fixed)**2
        q_amp_squeezed[i] = (var_n - n_val) / n_val
    else:
        q_amp_squeezed[i] = np.cosh(2 * r_fixed)

# Plotting: Multi-panel comparative analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Mandel Q vs Mean Photon Number
ax1.plot(n_sweep, q_fock, 'm-', lw=3, label=r'Fock State $|n\rangle$ ($Q=-1$, Minimum Bound)')
ax1.plot(n_sweep, q_amp_squeezed, color='#1f77b4', lw=2.5, ls='-',
         label=rf'Amplitude Squeezed ($r={r_fixed}$, $Q<0$)')
ax1.plot(n_sweep, q_coherent, 'g--', lw=2.2, label=r'Coherent State ($Q=0$, Shot Noise Limit)')
ax1.plot(n_sweep, q_thermal, 'r-.', lw=2.2, label=r'Thermal State ($Q=\bar{n}$)')
ax1.plot(n_sweep, q_sq_vac, 'orange', lw=2.2, ls=':', label=r'Squeezed Vacuum ($Q=1+2\bar{n}$)')

# Shaded Non-Classical vs Classical Regions
ax1.axhspan(-1.1, 0.0, color='lightgreen', alpha=0.25, label='Sub-Poissonian (Non-Classical)')
ax1.axhspan(0.0, 32.0, color='mistyrose', alpha=0.2, label='Super-Poissonian (Classical-Permitted)')

ax1.set_xlim(0, 15)
ax1.set_ylim(-1.2, 16)
ax1.set_xlabel(r"Mean Photon Number $\langle \hat{n} \rangle$")
ax1.set_ylabel(r"Mandel $Q$-Parameter")
ax1.set_title("Mandel $Q$-Parameter Across Canonical Quantum States")
ax1.legend(loc='upper left', framealpha=0.92, fontsize=9)
ax1.grid(True)

# Panel 2: Fano Factor F = Var(n) / <n>
ax2.plot(n_sweep, q_fock + 1.0, 'm-', lw=3, label=r'Fock State ($F=0$)')
ax2.plot(n_sweep, q_amp_squeezed + 1.0, color='#1f77b4', lw=2.5, label=r'Amplitude Squeezed ($F<1$)')
ax2.plot(n_sweep, q_coherent + 1.0, 'g--', lw=2.2, label=r'Coherent State ($F=1$)')
ax2.plot(n_sweep, q_thermal + 1.0, 'r-.', lw=2.2, label=r'Thermal State ($F=1+\bar{n}$)')
ax2.plot(n_sweep, q_sq_vac + 1.0, 'orange', lw=2.2, ls=':', label=r'Squeezed Vacuum ($F=2+2\bar{n}$)')

ax2.axhline(1.0, color='black', ls='--', lw=1.5)
ax2.set_xlim(0, 15)
ax2.set_ylim(-0.1, 17)
ax2.set_xlabel(r"Mean Photon Number $\langle \hat{n} \rangle$")
ax2.set_ylabel(r"Fano Factor $F = \mathrm{Var}(n) / \langle n \rangle$")
ax2.set_title("Fano Factor: Relative Noise Variance")
ax2.legend(loc='upper left', framealpha=0.92, fontsize=9)
ax2.grid(True)

plt.tight_layout()
plt.show()


## 3. Optimization of Sub-Poissonian Light: The Deep Squeezing Trade-off

For an amplitude-squeezed state $|\alpha, r\rangle$, how negative can $Q$ become?

Recall:
$$
Q(r, \alpha) = \frac{\alpha^2 (e^{-2r} - 1) + \sinh^2 r \cosh 2r}{\alpha^2 + \sinh^2 r}
$$

Let us analyze two extreme asymptotic limits:

1. **Weak Squeezing ($r \ll 1$):**
   Expanding in Taylor series for small $r$:
   $$
   e^{-2r} - 1 \approx -2r + 2r^2, \qquad \sinh^2 r \approx r^2
   $$
   $$
   Q \approx -2r + \mathcal{O}(r^2) < 0
   $$
   $Q$ becomes immediately negative as squeezing is switched on!

2. **Extreme Deep Squeezing ($r \gg 1$):**
   In this limit, $\sinh^2 r \approx \frac{1}{4} e^{2r}$ and $\cosh 2r \approx \frac{1}{2} e^{2r}$.
   The parametric pair noise term $\sinh^2 r \cosh 2r \sim \frac{1}{8} e^{4r}$ grows exponentially faster than any polynomial or coherent displacement!
   Therefore:
   $$
   \lim_{r \to \infty} Q(r, \alpha) = +\infty
   $$
   **Counter-intuitive Result:** Squeezing beyond a critical optimal value $r_{\text{opt}}$ destroys sub-Poissonian statistics and makes the beam fiercely super-Poissonian!

---

### Optimal Squeezing Parameter $r_{\text{opt}}$

For a fixed coherent intensity $\alpha^2$, differentiating $Q(r)$ with respect to $r$ yields the optimal squeezing level that minimizes $Q$:
$$
r_{\text{opt}} \approx \frac{1}{3} \ln(2\alpha) \quad (\alpha \gg 1)
$$
At this optimal point, the Mandel parameter reaches:
$$
Q_{\text{opt}} \approx -1 + \mathcal{O}(\alpha^{-2/3})
$$
approaching the ideal Fock state bound $Q = -1$ as coherent intensity becomes macroscopic!


In [ ]:
# Numerical Landscape: Mandel Q as a 2D Function of Displacement alpha and Squeezing r

alpha_grid = np.linspace(0.5, 6.0, 80)
r_grid = np.linspace(0.01, 1.8, 80)
A, R = np.meshgrid(alpha_grid, r_grid)

# Compute 2D grid of Mandel Q
n_bar_grid = A**2 + np.sinh(R)**2
var_grid = A**2 * np.exp(-2 * R) + 2 * np.sinh(R)**2 * np.cosh(R)**2
Q_grid = (var_grid - n_bar_grid) / n_bar_grid

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Contour Map of Mandel Q
contour = ax1.contourf(A, R, Q_grid, levels=np.linspace(-0.9, 1.5, 25), cmap='RdYlBu_r')
cbar = plt.colorbar(contour, ax=ax1, fraction=0.046, pad=0.04, label='Mandel Q')

# Zero contour (boundary between sub- and super-Poissonian)
ax1.contour(A, R, Q_grid, levels=[0.0], colors='black', linewidths=2.5, linestyles='--')

# Trace optimal r for each alpha
r_optimal = []
for a in alpha_grid:
    n_b = a**2 + np.sinh(r_grid)**2
    v = a**2 * np.exp(-2 * r_grid) + 2 * np.sinh(r_grid)**2 * np.cosh(r_grid)**2
    q = (v - n_b) / n_b
    r_optimal.append(r_grid[np.argmin(q)])
ax1.plot(alpha_grid, r_optimal, 'k-', lw=2.5, label=r'Optimal Squeezing $r_{\mathrm{opt}}(\alpha)$')

ax1.set_xlabel(r"Displacement Amplitude $\alpha$")
ax1.set_ylabel(r"Squeeze Parameter $r$")
ax1.set_title(r"2D Landscape: Mandel $Q(\alpha, r)$ (Dashed line: $Q=0$)")
ax1.legend(loc='upper left', framealpha=0.9)

# Panel 2: Slices across r for selected alpha values
for a_val in [1.5, 3.0, 5.0]:
    n_b = a_val**2 + np.sinh(r_grid)**2
    v = a_val**2 * np.exp(-2 * r_grid) + 2 * np.sinh(r_grid)**2 * np.cosh(r_grid)**2
    q = (v - n_b) / n_b
    min_q = np.min(q)
    min_r = r_grid[np.argmin(q)]
    ax2.plot(r_grid, q, lw=2.2, label=rf'$\alpha={a_val:.1f}$ (Min $Q={min_q:.2f}$ at $r={min_r:.2f}$)')

ax2.axhline(0.0, color='gray', ls='--', lw=1.5, label=r'Poissonian Limit ($Q=0$)')
ax2.axhline(-1.0, color='purple', ls=':', lw=1.5, label=r'Fock Bound ($Q=-1$)')

ax2.set_xlim(0, 1.8)
ax2.set_ylim(-1.05, 1.2)
ax2.set_xlabel(r"Squeeze Parameter $r$")
ax2.set_ylabel(r"Mandel $Q$-Parameter")
ax2.set_title("Squeezing Optimization: The Non-Monotonic Minimum")
ax2.legend(loc='upper right', framealpha=0.9)
ax2.grid(True)

plt.tight_layout()
plt.show()


> **Physical Insight: Why Can't $Q$ Drop Below $-1$?**
> The mathematical lower bound $Q \ge -1$ is fundamentally rooted in the Born probability rule:
> $$
> \text{Var}(n) = \langle \hat{n}^2 \rangle - \langle \hat{n} \rangle^2 \ge 0
> $$
> In any valid quantum state, variance is an expectation value of a positive semidefinite operator $(\hat{n} - \langle \hat{n} \rangle)^2$, and therefore cannot be negative.
>
> Setting $\text{Var}(n) = 0$ gives $F = 0$ and $Q = -1$. This zero-fluctuation condition is satisfied *if and only if* the field is in an exact Fock number eigenstate $|n\rangle$ ($n \ge 1$). A Fock state represents the ultimate limit of photon noise suppression: absolute determinism in photon counting!


In [ ]:
# Interactive Exploration: Mandel Q Explorer Across State Families

def explore_mandel_q(state_type='Displaced Squeezed', param1=3.0, param2=0.5):
    """Interactive visualization comparing photon distribution and Mandel Q."""
    N = 45
    n_axis = np.arange(N)
    
    if state_type == 'Displaced Squeezed':
        # param1 = alpha, param2 = r
        alpha, r = param1, param2
        # Amplitude squeezed along X1 (theta = 0)
        vac = np.zeros(N, dtype=np.complex128)
        vac[0] = 1.0
        # annihilation operator
        a = np.diag(np.sqrt(np.arange(1, N)), k=1)
        S = la.expm(0.5 * (r * a@a - r * a.T.conj()@a.T.conj()))
        D = la.expm(alpha * a.T.conj() - alpha * a)
        psi = D @ (S @ vac)
        p_n = np.abs(psi)**2
        stats = quantum_moments(psi, N)
        title_str = rf"Displaced Squeezed ($\alpha={alpha:.1f}, r={r:.2f}$)"
        
    elif state_type == 'Thermal':
        # param1 = n_bar
        n_bar = param1
        p_n = (1.0 / (1.0 + n_bar)) * ((n_bar / (1.0 + n_bar)) ** n_axis)
        rho = np.diag(p_n).astype(np.complex128)
        stats = quantum_moments(rho, N)
        title_str = rf"Thermal State ($\bar{{n}}={n_bar:.1f}$)"
        
    elif state_type == 'Coherent':
        # param1 = alpha
        alpha = param1
        p_n = np.array([(alpha**(2*n)) * np.exp(-alpha**2) / math.factorial(n) for n in n_axis])
        psi = np.sqrt(p_n).astype(np.complex128)
        stats = quantum_moments(psi, N)
        title_str = rf"Coherent State ($|\alpha|={alpha:.1f}$)"
        
    else:  # Fock
        n_fock = int(param1)
        psi = np.zeros(N, dtype=np.complex128)
        if n_fock < N:
            psi[n_fock] = 1.0
        p_n = np.abs(psi)**2
        stats = quantum_moments(psi, N)
        title_str = rf"Fock State $|{n_fock}\rangle$"
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 1. Photon distribution
    n_plot = min(N, int(stats['mean_n'] * 2.5 + 8))
    bars = ax1.bar(n_axis[:n_plot], p_n[:n_plot], color='#1f77b4', alpha=0.75, edgecolor='black', width=0.8)
    ax1.axvline(stats['mean_n'], color='red', ls='--', lw=2, label=rf"$\langle n \rangle = {stats['mean_n']:.2f}$")
    ax1.set_xlim(-0.5, n_plot)
    ax1.set_xlabel("Photon Number n")
    ax1.set_ylabel("P(n)")
    ax1.set_title(title_str)
    ax1.legend(loc='upper right')
    ax1.grid(True)
    
    # 2. Mandel Q Gauge Meter
    q_val = stats['mandel_q']
    f_val = stats['fano_factor']
    
    labels = [r'Mandel $Q$', r'Fano $F$', r'Variance $\mathrm{Var}(n)$', r'Mean $\langle n \rangle$']
    vals = [q_val, f_val, stats['var_n'], stats['mean_n']]
    colors = ['#d62728' if q_val > 0.05 else ('#2ca02c' if q_val < -0.05 else 'gray'),
              '#1f77b4', '#ff7f0e', '#9467bd']
              
    ax2.bar(labels, vals, color=colors, alpha=0.85, edgecolor='black', width=0.55)
    ax2.axhline(0.0, color='black', lw=1.2)
    for i, v in enumerate(vals):
        ax2.text(i, v + (0.1 if v >= 0 else -0.3), f"{v:+.2f}", ha='center', va='bottom' if v >= 0 else 'top', fontweight='bold')
        
    regime = "Sub-Poissonian (Quantum)" if q_val < -0.05 else ("Super-Poissonian (Classical)" if q_val > 0.05 else "Poissonian (Shot Noise)")
    ax2.set_title(f"Non-Classicality Witness: {regime}")
    ax2.grid(True, axis='y')
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_mandel_q,
             state_type=Dropdown(options=['Displaced Squeezed', 'Coherent', 'Thermal', 'Fock'], value='Displaced Squeezed'),
             param1=FloatSlider(min=0.5, max=5.0, step=0.5, value=3.0, description='Param 1:'),
             param2=FloatSlider(min=0.0, max=1.2, step=0.1, value=0.5, description='Squeezing r:'))
else:
    # Static execution fallback for headless environments
    explore_mandel_q(state_type='Displaced Squeezed', param1=3.0, param2=0.5)


## 4. Self-Check Exercises and Pedagogical Challenges

Consolidate your mastery of the Mandel $Q$-parameter and quantum non-classicality criteria through these analytical and numerical challenges.

---

### Challenge 1: The Mandel Parameter of Schrödinger Cat States
An **even Schrödinger cat state** is an equal superposition of coherent states with opposite phases:
$$
|\psi_{\text{even}}\rangle = \mathcal{N}_+ \left( |\alpha\rangle + |-\alpha\rangle \right)
$$
where $\mathcal{N}_+ = \left[ 2(1 + e^{-2|\alpha|^2}) \right]^{-1/2}$.
1. Expand $|\psi_{\text{even}}\rangle$ in the Fock basis and explain why only even Fock components ($|0\rangle, |2\rangle, |4\rangle \dots$) have non-zero probability.
2. Given that $\langle \hat{n} \rangle = |\alpha|^2 \tanh(|\alpha|^2)$ and $\langle \hat{n}^2 \rangle = |\alpha|^2 \tanh(|\alpha|^2) + |\alpha|^4$, show that the Mandel $Q$-parameter is:
   $$
   Q = |\alpha|^2 \left[ 1 - \tanh(|\alpha|^2) \right] = \frac{2|\alpha|^2}{e^{2|\alpha|^2} + 1} > 0
   $$
3. Is an even cat state sub-Poissonian or super-Poissonian? Is it classical or non-classical?

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
Recalling $|\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty \frac{\alpha^n}{\sqrt{n!}} |n\rangle$ and $|-\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty \frac{(-\alpha)^n}{\sqrt{n!}} |n\rangle$:
$$
|\alpha\rangle + |-\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty \frac{\alpha^n [1 + (-1)^n]}{\sqrt{n!}} |n\rangle
$$
For odd $n$, $1 + (-1)^n = 0$. For even $n = 2m$, $1 + (-1)^{2m} = 2$.
Therefore, the superposition contains *exclusively even Fock states* ($P_{2m+1} \equiv 0$).

**Part 2 Solution:**
The photon variance is:
$$
\text{Var}(n) = \langle \hat{n}^2 \rangle - \langle \hat{n} \rangle^2 = |\alpha|^2 \tanh(|\alpha|^2) + |\alpha|^4 [1 - \tanh^2(|\alpha|^2)]
$$
The Mandel parameter is:
$$
Q = \frac{\text{Var}(n) - \langle n \rangle}{\langle n \rangle} = \frac{|\alpha|^4 [1 - \tanh^2(|\alpha|^2)]}{|\alpha|^2 \tanh(|\alpha|^2)} = |\alpha|^2 \frac{\text{sech}^2(|\alpha|^2)}{\tanh(|\alpha|^2)} = \frac{|\alpha|^2}{\sinh(|\alpha|^2)\cosh(|\alpha|^2)} = \frac{2|\alpha|^2}{\sinh(2|\alpha|^2)} > 0
$$

**Part 3 Solution:**
Because $Q > 0$ for all finite $\alpha$, the even cat state is **super-Poissonian**!
However, it is **profoundly non-classical**: it exhibits quantum macroscopic superposition, negative Wigner function values, and quantum interference fringes. This is another vivid example where $Q > 0$ does *not* imply classicality!
</details>

---

### Challenge 2: Numerical Verification with QuTiP
Using QuTiP, compute the Mandel $Q$ parameter for an amplitude-squeezed state with $\alpha = 3.0$ and $r = 0.5$, and verify that $Q$ matches our NumPy implementation to within $< 10^{-6}$.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
if HAS_QUTIP:
    import qutip as qt
    N = 60
    alpha = 3.0
    r = 0.5
    
    # QuTiP state synthesis
    vac = qt.fock(N, 0)
    psi_qt = qt.displace(N, alpha) * qt.squeeze(N, r) * vac
    
    # Operators
    n_op = qt.num(N)
    mean_n = qt.expect(n_op, psi_qt)
    mean_n2 = qt.expect(n_op**2, psi_qt)
    var_n = mean_n2 - mean_n**2
    q_qt = (var_n - mean_n) / mean_n
    
    print(f"QuTiP Mandel Q: {q_qt:.6f}")
    assert q_qt < 0.0, "State should be sub-Poissonian!"
    print("QuTiP validation confirmed sub-Poissonian light!")
else:
    print("QuTiP not installed; run in Google Colab to execute this test.")
```
</details>


## 5. Summary, Reference Table, and Outlook

### Comprehensive Comparison: Non-Classicality and Mandel Q

| State | Mean $\langle \hat{n} \rangle$ | Variance $\text{Var}(n)$ | Mandel $Q$ | Fano $F$ | Non-Classical? |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Fock State $|n\rangle$** | $n$ | $0$ | $-1.0$ | $0.0$ | **Yes** (Ultimate Quantum) |
| **Coherent State $|\alpha\rangle$** | $|\alpha|^2$ | $|\alpha|^2$ | $0.0$ | $1.0$ | **No** (Classical Benchmark) |
| **Thermal State $\hat{\rho}_{\text{th}}$** | $\bar{n}$ | $\bar{n} + \bar{n}^2$ | $+\bar{n} > 0$ | $1+\bar{n} > 1$ | **No** (Classical Mixture) |
| **Squeezed Vacuum $|0, r\rangle$** | $\sinh^2 r$ | $2\bar{n}(\bar{n}+1)$ | $1+2\bar{n} > 0$ | $2+2\bar{n} > 2$ | **Yes** (Quadrature squeezed) |
| **Amp. Squeezed $|\alpha, r\rangle$** | $\alpha^2 + \sinh^2 r$ | $\alpha^2 e^{-2r} + \dots$ | **$< 0$** | **$< 1$** | **Yes** (Sub-Poissonian) |
| **Even Cat State $|\psi_+\rangle$** | $|\alpha|^2\tanh|\alpha|^2$ | $> \langle n \rangle$ | $> 0$ | $> 1$ | **Yes** (Wigner negative) |

---

### Key Takeaways
1. **$Q < 0$ is Strictly Quantum:** Any optical state with $Q < 0$ (or $F < 1$) has a non-positive Glauber-Sudarshan $P(\alpha)$ distribution and cannot be described by classical electrodynamics.
2. **Absolute Lower Bound ($Q \ge -1$):** Because quantum variance $\text{Var}(n) \ge 0$, $Q$ cannot be less than $-1$. The lower bound is achieved exclusively by Fock number states $|n\rangle$.
3. **The Squeezing Trade-off:** Squeezing initially suppresses photon noise ($Q < 0$), but extreme squeezing ($r \to \infty$) causes parametric photon pair noise to dominate, driving $Q \to +\infty$.

---

### Primary References
- **Mandel, L.** (1979). *Sub-Poissonian photon statistics in resonance fluorescence*. Optics Letters, 4(7), 205–207.
- **Short, R., & Mandel, L.** (1983). *Observation of sub-Poissonian photon statistics*. Physical Review Letters, 51(5), 384–387.
- **Gerry, C. C., & Knight, P. L.** (2005). *Introductory Quantum Optics*. Cambridge University Press. Chapter 4: Non-classical Properties of Single-Mode Light.
- **Walls, D. F., & Milburn, G. J.** (2008). *Quantum Optics*. Springer. Chapter 3: Representations of the Electromagnetic Field.

---

### Next in the Curriculum
In **Notebook 1.7: Second-Order Coherence and Photon Anti-Bunching**, we complete Module 1 by examining the intensity-intensity correlation function $g^{(2)}(\tau)$, proving why photon anti-bunching ($g^{(2)}(0) < 1$) and the Hanbury Brown and Twiss effect form the experimental bedrock of modern quantum optics.
